# 01 · CNN + regresyon: deneme eğitimi

Kurulum (00 ile aynı) → mimari kontrolü → küçük-batch ezber testi → **rejim A, seed 0 deneme eğitimi** → kayıp eğrileri → doğrulama testi (tek dairede hata < %3).

**Önce:** *Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU*. Tüm not defteri ≈ 6–8 dakika.

In [ ]:
import torch, platform
print("PyTorch", torch.__version__, "| Python", platform.python_version())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU YOK → Çalışma zamanı > Çalışma zamanı türünü değiştir > T4 GPU (veri kontrolleri yine de çalışır)")

## 1. Proje klasörü
`GITHUB_URL`'yi ortak reponuzun adresiyle değiştirin. Repo private ise token Colab Secrets'tan (`GH_TOKEN`) okunur; public ise hiçbir şey yapmanız gerekmez. Sonuçlar her iki durumda da Drive'a yazılır (Colab kapanınca kaybolmasın diye).

In [ ]:
KAYNAK = "github"                                          # "github" ya da "drive"
GITHUB_URL = "https://github.com/KULLANICI/ebbinghaus-human-ai.git"      # ortak reponuzun adresi
DAL = "main"                                              # PR birleşmeden test için: "veri-sinifi"
DRIVE_PROJE = "/content/drive/MyDrive/ebbinghaus-proje"   # yalnızca KAYNAK = "drive" ise

import os, sys
from google.colab import drive, userdata
drive.mount("/content/drive")

if KAYNAK == "github":
    # Repo private ise: sol menü 🔑 Secrets → GH_TOKEN adıyla token ekleyin.
    # Public ise token gerekmez, bu adım atlanır.
    try:
        token = userdata.get("GH_TOKEN")
        url = GITHUB_URL.replace("https://", f"https://{token}@")
    except Exception:
        url = GITHUB_URL
    PROJE = "/content/" + GITHUB_URL.rstrip("/").split("/")[-1].removesuffix(".git")
    if not os.path.exists(PROJE):
        !git clone -q -b {DAL} {url} {PROJE}
    else:
        !git -C {PROJE} fetch -q origin {DAL} && git -C {PROJE} checkout -q {DAL} && git -C {PROJE} pull -q origin {DAL}
    SONUC = "/content/drive/MyDrive/ebbinghaus-sonuclar/results"
else:
    PROJE = DRIVE_PROJE
    SONUC = os.path.join(PROJE, "results")

os.makedirs(SONUC, exist_ok=True)
os.chdir(PROJE)
if PROJE not in sys.path:
    sys.path.insert(0, PROJE)
print("Proje :", PROJE, "→", sorted(os.listdir(PROJE)))
print("Sonuç :", SONUC)

## 2. Ortak uyaran üreteci (Kişi 1'in `stimuli.py`'si)

In [ ]:
import stimuli
print("MODEL ayarları:", stimuli.MODEL)
for (kurulum, kosul), e in stimuli.check_all().items():
    print(f"{kurulum:6s} {kosul:10s} dış sınır {e:6.1f} px")
print("stimuli.check_all ✓ — tüm koşullar tuval içinde")

## Veri sınıfı kontrolleri

In [ ]:
from model import data
data.check_all(verbose=False); print("Veri kontrolleri ✓")

## 1. Mimari kontrolü
Beklenen: ≈ 0,55 M parametre, kuramsal alıcı alan 94 px; deneysel alıcı alan 94'ü aşmamalı ve ona yakın olmalı.

In [ ]:
import torch
from model.cnn import RegressionNet, TwoAFCNet, n_params, receptive_field, receptive_field_empirical
from model.train import get_device, set_seed
device = get_device()

set_seed(0)
net = RegressionNet().to(device)
print(net)
print(f"\nParametre: {n_params(net):,}  (2AFC siyam: {n_params(TwoAFCNet()):,})")
x = torch.zeros(4, 1, 224, 224, device=device)
print("Çıktı boyutu:", tuple(net(x).shape), "| gövde çıktısı:", tuple(net.encoder(x).shape))
rf_t = receptive_field()
rf_e = receptive_field_empirical(net.encoder)
print(f"Alıcı alan: kuramsal {rf_t} px, deneysel {rf_e[0]}×{rf_e[1]} px")
assert max(rf_e) <= rf_t, "Deneysel alıcı alan kuramsal değeri aşıyor — mimariyi kontrol edin"

## 2. Ezber testi (hata ayıklama için)
Model tek bir 64'lük batch'i ezberleyebilmeli. Etiket varyansı ≈ 0,053; 200 adımda MSE 0,005'in altına (≥ 10 kat düşüş) inmezse döngüde ya da veride bir hata vardır.

In [ ]:
from torch.utils.data import DataLoader
set_seed(0)
m = RegressionNet().to(device)
opt = torch.optim.Adam(m.parameters(), lr=1e-3)
xb, yb = next(iter(DataLoader(data.EbbinghausDataset("B-fiz", "val", n=64, seed=0), batch_size=64)))
xb, yb = xb.to(device), yb.to(device)
for step in range(1, 201):
    m.train()
    loss = torch.nn.functional.mse_loss(m(xb), yb)
    opt.zero_grad(); loss.backward(); opt.step()
    if step in (1, 50, 100, 200):
        print(f"adım {step:3d}  MSE {loss.item():.5f}")
print("Ezber testi", "✓" if loss.item() < 5e-3 else "✗ — bana çıktıyı gönderin")
del m, opt

## 3. Deneme eğitimi: rejim A, seed 0
Plandaki ayarlar: 10 epoch × 20.000 görsel, doğrulama 2.000, Adam 1e-3, batch 64. Ağırlıklar ve epoch kaydı `SONUC` klasörüne yazılır.
Hızlı denemek isterseniz `EPOCHS = 3` yapın.

In [ ]:
from model.train import train_regression
EPOCHS = 10
net, hist = train_regression("A", seed=0, out_dir=SONUC, epochs=EPOCHS)

## 4. Kayıp eğrileri

In [ ]:
import matplotlib.pyplot as plt
ep = [h["epoch"] for h in hist]
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
ax[0].plot(ep, [h["train_mse"] for h in hist], "o-", label="eğitim")
ax[0].plot(ep, [h["val_mse"] for h in hist], "s-", label="doğrulama")
ax[0].set_yscale("log"); ax[0].set_xlabel("epoch"); ax[0].set_ylabel("MSE"); ax[0].legend()
ax[0].set_title("Kayıp (rejim A, seed 0)")
ax[1].plot(ep, [h["val_mape"] for h in hist], "s-", color="C1")
ax[1].axhline(3, ls="--", color="gray"); ax[1].text(ep[0], 3.1, "%3 eşiği", color="gray")
ax[1].set_xlabel("epoch"); ax[1].set_ylabel("ortalama mutlak hata (%)"); ax[1].set_title("Doğrulama hatası")
plt.tight_layout()
plt.savefig(f"{SONUC}/model_regresyon_A_seed0_kayip.png", dpi=150)
plt.show()

## 5. Doğrulama testi (plan §5)
Tek daire, insanla aynı 5 karşılaştırma boyutu (D × 0.84 … 1.16), test ile aynı çizim yolu (`stimuli.draw_config`). Ortalama mutlak hata **< %3** olmalı; geçmeyen model yorumlanmaz.

In [ ]:
from model.train import single_circle_check
overall, ok, per = single_circle_check(net)
for c, v in per.items():
    print(f"  D × {c:.2f}: ortalama tahmin {v['mean_pred']:.3f}   hata %{v['mape']:.2f}")
print(f"Genel hata %{overall:.2f} → {'GEÇTİ ✓' if ok else 'GEÇMEDİ ✗'}")

## Gözden geçiren (Kişi 1) için kontrol listesi — yaklaşık 15 dk
- [ ] Mimari plandakiyle aynı: 5 blok (32-64-128-128-256), BN + ReLU + MaxPool, global ortalama havuzlama, ≈ 0,55 M parametre.
- [ ] Deneysel alıcı alan ≤ 94 px.
- [ ] Ezber testi ✓.
- [ ] Kayıp eğrileri düşüyor; doğrulama eğitimden belirgin şekilde ayrışmıyor.
- [ ] Doğrulama testi geçti (tek dairede hata < %3).
- [ ] `SONUC` klasöründe `model_regresyon_A_seed0.pt`, `_log.csv` ve `_kayip.png` var.